# 01. Разведочный анализ данных и strict contract

**Цель раздела.** Показать, из чего состоит `raw_data`, как строится строгий оконный контракт и почему дальше важно оценивать workload, горизонты предупреждения и разделение `Норма / Предупреждение / Риск`.

Этот notebook не обучает финальную модель. Он готовит постановку данных для baseline и Efficient HADES.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "dashboard" / "artifacts"
CONTRACT_PATH = ARTIFACT_DIR / "contract" / "strict_intensity_contract.csv.gz"
REPORT_DIR = ARTIFACT_DIR / "reports"
BASELINE_DIR = REPORT_DIR / "protocol_fixed_baselines"
MODEL_OUT_DIR = ARTIFACT_DIR / "model" / "protocol_fixed_efficient_hades"
RUN_ROOT = ARTIFACT_DIR / "model_runs" / "protocol_fixed_hades"
RUN_ID = "final_protocol_fixed_hades"
BUNDLE_DIR = ARTIFACT_DIR / "bundle"

def run_command(command: list[str]) -> None:
    print(" ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


## 1. Источник данных

**Цель раздела.** Проверить, что финальный контур начинается только с `raw_data`, а не со старых промежуточных файлов.


In [ ]:
RAW_DATA = PROJECT_ROOT / "raw_data"
rows = []
for workload in sorted([p for p in RAW_DATA.iterdir() if p.is_dir()]) if RAW_DATA.exists() else []:
    log_files = list((workload / "mapred_logs").glob("*.log"))
    kpi_files = [p for p in (workload / "kpis").rglob("*") if p.is_file()] if (workload / "kpis").exists() else []
    rows.append({"workload": workload.name, "log_files": len(log_files), "kpi_files": len(kpi_files)})
raw_inventory = pd.DataFrame(rows).sort_values("workload")
display(raw_inventory)
ax = raw_inventory.set_index("workload")[["log_files", "kpi_files"]].plot(kind="bar", figsize=(10, 4), title="Состав raw_data по workload")
ax.set_xlabel("Workload")
ax.set_ylabel("Количество файлов")
plt.tight_layout()


**Локальный вывод.** Таблица фиксирует физический источник данных. Неравномерность workload учитывается в дальнейшем протоколе оценки, чтобы качество отражало обнаружение аномалий, а не только различие доменов.


## 2. Построение strict contract

**Цель раздела.** Получить единую оконную таблицу с логами, KPI, split и целевыми метками.


In [ ]:
if not CONTRACT_PATH.exists():
    run_command([sys.executable, "-m", "dashboard.pipeline.build_contract", "--raw-data", "raw_data"])

contract = pd.read_csv(CONTRACT_PATH, low_memory=False)
contract["window_ts"] = pd.to_datetime(contract["window_ts"], utc=True, errors="coerce")
if "y_anomaly" not in contract.columns and {"y_warning", "y_risk"}.issubset(contract.columns):
    contract["y_anomaly"] = np.maximum(contract["y_warning"].astype(int), contract["y_risk"].astype(int))
display(contract.head(8))
display(contract.groupby("split").agg(rows=("run_id", "size"), runs=("run_id", "nunique"), workloads=("workload", "nunique")).reset_index())


**Локальный вывод.** Contract stage задает честные границы окон: признаки берутся из окна наблюдения, а метки предупреждения и риска считаются с gap перед событием.


## 3. Разметка и горизонты

**Цель раздела.** Показать, что `y_warning`, `y_risk` и horizon-цели решают разные задачи.


In [ ]:
target_cols = [c for c in ["y_warning", "y_risk", "y_anomaly", "horizon_30s", "horizon_60s", "horizon_120s", "horizon_300s"] if c in contract.columns]
if target_cols:
    target_summary = contract.groupby("split")[target_cols].mean().T * 100
    display(target_summary.round(2).rename_axis("target").reset_index())
    ax = target_summary.plot(kind="bar", figsize=(11, 4), title="Доля positive-окон по split, %")
    ax.set_xlabel("Target")
    ax.set_ylabel("Positive windows, %")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
else:
    print("Horizon columns будут построены в protocol-fixed training module.")


**Локальный вывод.** Дальние горизонты обычно лучше отделимы. `horizon_60s` полезен как ранний канал, но он шумнее.


## 4. Workload/domain shift

**Цель раздела.** Проверить, не прячется ли проблема качества в split и workload-domain shift.


In [ ]:
cols = ["workload", "split"] + [c for c in ["y_warning", "y_risk", "y_anomaly"] if c in contract.columns]
summary = contract[cols].groupby(["workload", "split"]).agg(
    rows=("split", "size"),
    warning_rate=("y_warning", "mean") if "y_warning" in contract.columns else ("split", "size"),
    risk_rate=("y_risk", "mean") if "y_risk" in contract.columns else ("split", "size"),
).reset_index()
display(summary)
ax = summary.pivot(index="workload", columns="split", values="rows").fillna(0).plot(kind="bar", stacked=True, figsize=(10, 4), title="Окна по workload и split")
ax.set_xlabel("Workload")
ax.set_ylabel("Количество окон")
plt.tight_layout()


**Локальный вывод.** Поэтому финальная оценка использует `stratified_by_workload`: каждый workload представлен в train, validation и test, а сравнение моделей остается устойчивым к доменной структуре данных.


## 5. Доступность модальностей

**Цель раздела.** Проверить, насколько часто в окне есть логи, KPI и обе модальности одновременно.


In [ ]:
availability_cols = [c for c in ["log_available", "kpi_available", "kpi_series_available", "has_both_modalities"] if c in contract.columns]
if availability_cols:
    availability = contract.groupby("split")[availability_cols].mean().T * 100
    display(availability.round(2).rename_axis("feature").reset_index())
    ax = availability.plot(kind="bar", figsize=(10, 4), title="Доступность логов и KPI по split, %")
    ax.set_xlabel("Признак доступности")
    ax.set_ylabel("Окна, %")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()


**Локальный вывод.** Fusion-модель должна работать с неполными окнами, а dashboard должен отдельно показывать score модели, alert-layer и факт incident.

## Итог 01

Этап `01` фиксирует исходные данные и строгий контракт задачи. Следующий шаг — `02_baseline_pipeline.ipynb`, где на этом же контракте строится контрольная baseline-линия для дальнейшего сравнения с финальной моделью.
